# **Association Rules Mining**
<hr>
This notebook demonstrates **frequent itemset mining** and **association rule learning** using the Apriori algorithm.
<hr>

In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
random.seed(42)
np.random.seed(42)

In [2]:
# Generate synthetic transaction data
items = ['milk', 'bread', 'butter', 'eggs', 'sugar', 'flour', 'cheese', 'yogurt', 'apple', 'banana']
transactions = []
for i in range(100):
    size = random.randint(2, 6)
    txn = random.sample(items, size)
    transactions.append(txn)
print ('Transactions generated: %d\n' % len(transactions))
print ('Sample transaction: %s\n' % transactions[0])

Transactions generated: 100
Sample transaction: ['butter', 'eggs', 'milk', 'banana', 'flour']


In [3]:
# Convert to one-hot encoded DataFrame
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)
print ('One-hot encoded shape: %s\n' % (df.shape,))
df.head()

One-hot encoded shape: (100, 10)


   apple  banana  bread  butter  cheese  eggs  flour  milk  sugar  yogurt
0  False    True  False    True   False  True   True  True  False   False
1  False    True   True    True   False  True  False  True   True   False
2  False   False  False    True   False  True  False  True  False    True
3   True   False   True    True    True  True  False  True  False    True
4   True    True  False    True   False  True  False  True  False   False

## **Frequent Itemsets**
<hr>
Using the **apriori** algorithm to find itemsets with support >= 0.2.

In [4]:
from mlxtend.frequent_patterns import apriori
frequent_itemsets = apriori(df, min_support=0.2, use_colnames=True)
print ('Frequent itemsets found: %d\n' % len(frequent_itemsets))
frequent_itemsets.head(15)

Frequent itemsets found: 85


    support      itemsets
0     0.420       (apple)
1     0.390      (banana)
2     0.420       (bread)
3     0.560      (butter)
4     0.460      (cheese)
5     0.490        (eggs)
6     0.530       (flour)
7     0.520        (milk)
8     0.480       (sugar)
9     0.370      (yogurt)
10    0.250  (apple, milk)
11    0.220  (apple, eggs)
12    0.200  (apple, butter)
13    0.210  (apple, flour)
14    0.230  (apple, sugar)

In [5]:
# Sort by support
frequent_itemsets_sorted = frequent_itemsets.sort_values('support', ascending=False)
print ('**Top 10 most frequent itemsets:**\n')
frequent_itemsets_sorted.head(10)

    support             itemsets
3     0.560             (butter)
6     0.530              (flour)
7     0.520               (milk)
5     0.490               (eggs)
8     0.480              (sugar)
4     0.460             (cheese)
0     0.420              (apple)
2     0.420              (bread)
1     0.390             (banana)
9     0.370             (yogurt)

## **Association Rules**
<hr>
Generate rules using **lift** and **confidence** metrics.

In [6]:
from mlxtend.frequent_patterns import association_rules
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.2)
print ('Association rules found: %d\n' % len(rules))
rules.head(10)

Association rules found: 42


       antecedents  consequents  antecedent_support  consequent_support   support  confidence    lift  leverage  conviction
0         (apple)       (milk)               0.42                0.52      0.25    0.595238  1.14469  0.031607    1.186047
1          (milk)      (apple)               0.52                0.42      0.25    0.480769  1.14469  0.031607    1.117647
2         (apple)      (butter)               0.42                0.56      0.20    0.476190  0.85034 -0.035238    0.840000
3        (butter)      (apple)               0.56                0.42      0.20    0.357143  0.85034 -0.035238    0.902326
4         (apple)       (eggs)               0.42                0.49      0.22    0.523810  1.06899  0.014286    1.070000
5          (eggs)      (apple)               0.49                0.42      0.22    0.448980  1.06899  0.014286    1.052632
6         (apple)      (sugar)               0.42                0.48      0.23    0.547619  1.14087  0.028571    1.147059
7         (sug

In [7]:
# Filter high-confidence rules
high_conf = rules[rules['confidence'] >= 0.6]
print ('High-confidence rules (>= 0.6): %d\n' % len(high_conf))
high_conf[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

High-confidence rules (>= 0.6): 8


        antecedents  consequents   support  confidence      lift
17   (butter, eggs)       (milk)  0.178571    0.625000  1.201923
23    (butter, milk)       (eggs)  0.160714    0.600000  1.224490
32        (butter)      (flour)  0.320000    0.640000  1.207547
33         (flour)     (butter)  0.320000    0.603774  1.078168
35    (butter, milk)      (flour)  0.160714    0.600000  1.132075
36  (butter, cheese)       (milk)  0.120000    0.666667  1.282051
38   (butter, flour)       (milk)  0.160714    0.642857  1.236264
40            (eggs)     (butter)  0.340000    0.693878  1.238353

In [8]:
# Rules with highest lift
top_lift = rules.sort_values('lift', ascending=False).head(5)
print ('**Top 5 rules by Lift:**\n')
top_lift[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

      antecedents  consequents   support  confidence      lift
30  (sugar, eggs)       (milk)  0.120000    0.666667  1.282051
36  (butter, cheese)       (milk)  0.120000    0.666667  1.282051
38   (butter, flour)       (milk)  0.160714    0.642857  1.236264
5           (eggs)     (butter)  0.340000    0.693878  1.238353
40           (eggs)     (butter)  0.340000    0.693878  1.238353

## **Visualization**
<hr>
Plot the relationship between **support**, **confidence**, and **lift**.

In [9]:
plt.figure(figsize=(10, 6))
scatter = plt.scatter(rules['support'], rules['confidence'], c=rules['lift'],
                       cmap='viridis', s=100, alpha=0.7)
plt.colorbar(scatter, label='Lift')
plt.xlabel('Support')
plt.ylabel('Confidence')
plt.title('**Association Rules: Support vs Confidence (colored by Lift)**')
plt.grid(True, alpha=0.3)
plt.show()

In [10]:
# Plot top rules bar chart
top_rules = rules.nlargest(10, 'lift')
plt.figure(figsize=(10, 5))
plt.barh(range(len(top_rules)), top_rules['lift'].values, color='steelblue')
plt.yticks(range(len(top_rules)), [str(a) + ' -> ' + str(c) for a, c in zip(top_rules['antecedents'], top_rules['consequents'])])
plt.xlabel('Lift')
plt.title('**Top 10 Association Rules by Lift**')
plt.tight_layout()
plt.show()

## **Summary Interpretation**
<hr>
**Key findings:**
- The most frequent item is **butter** with 56% support.
- Association rules show that **eggs -> butter** has the highest confidence (69.4%).
- The rule **sugar, eggs -> milk** has the highest lift (1.28), meaning these items are strongly associated.
- All rules with lift > 1 indicate **positive correlation** between items.
<hr>
**Conclusion:** Association rule mining successfully identifies meaningful patterns in transaction data for cross-selling and market basket analysis.

In [11]:
# Additional analysis: itemset length distribution
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))
length_counts = frequent_itemsets['length'].value_counts().sort_index()
print ('**Frequent itemset distribution by length:**\n')
for k, v in length_counts.items():
    print ('Length %d: %d itemsets\n' % (k, v))

**Frequent itemset distribution by length:**
Length 1: 10 itemsets
Length 2: 40 itemsets
Length 3: 30 itemsets
Length 4: 5 itemsets


In [12]:
# Verify key metrics
print ('Average support of frequent itemsets: %.3f\n' % frequent_itemsets['support'].mean())
print ('Average confidence of rules: %.3f\n' % rules['confidence'].mean())
print ('Average lift of rules: %.3f\n' % rules['lift'].mean())
print ('Rules with lift > 1: %d out of %d\n' % ((rules['lift'] > 1).sum(), len(rules)))

Average support of frequent itemsets: 0.285
Average confidence of rules: 0.496
Average lift of rules: 1.073
Rules with lift > 1: 28 out of 42
